In [0]:
%pip install --quiet h2o interpret h2o_pysparkling_3.5
#%%sh pip install --quiet h2o

In [0]:
pip install pls_common_data_store

In [0]:
#dbutils.library.restartPython()

In [0]:
import h2o
from h2o.automl import H2OAutoML
from pysparkling import H2OContext
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, log_loss, confusion_matrix, ConfusionMatrixDisplay
from interpret.glassbox import ExplainableBoostingClassifier
from interpret import show
import numpy as np
import pandas as pd
import pyspark.sql.functions as f
import pickle
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.clustering import KMeans
from pysparkling import H2OContext
from sklearn.impute import KNNImputer
from h2o.automl import H2OAutoML
from effodata import ACDS, golden_rules, Joiner, Sifter, Equality
from pls_common_data_store import pls_data_store
from kpi_metrics import KPI, AliasMetric, CustomMetric, AliasGroupby, Rollup, Cube, available_metrics, get_metrics
from scipy import stats
from scipy.stats import chi2_contingency

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", None)

In [0]:
print(spark.conf.get("spark.executor.memory"))
print(spark.conf.get("spark.executor.memoryOverhead", "not set"))
print(spark.conf.get("spark.dynamicAllocation.enabled", "not set"))
print(spark.conf.get("spark.executor.instances", "not set"))
print(sc._jsc.sc().getExecutorMemoryStatus().size()) 

In [0]:
use_sample_mart = True
acds = ACDS(use_sample_mart = use_sample_mart)
kpi_session = KPI(use_sample_mart = use_sample_mart)
pls_data_store_session = pls_data_store(spark)

# Holiday start and end dates
start_date = "2025-11-08"
end_date = "2026-12-25"

#### H2o Model Training: HH prediction for Halloween 2025

In [0]:
# going to split the train data into train test .. since cluster timed out in prev. split
halloween_data = spark.read.csv(
    'abfss://data-shuttle@sa8451denblrdatamoverdev.dfs.core.windows.net/d199104/lookalike/acquisition_lookalike__halloween_2025_trainingdata',
    header=True
)

halloween_data = halloween_data.withColumn("target_label", f.col("target_label").cast("int"))

train_fraction = 0.8
fractions = {0: train_fraction, 1: train_fraction}

model_train_data_final = halloween_data.sampleBy("target_label", fractions=fractions, seed=42)

test_data_final = halloween_data.join(
    model_train_data_final.select("ehhn"), on="ehhn", how="left_anti"
)

model_train_data_final.groupBy("target_label").agg(
    f.count("*").alias("hh_count"),
    f.round((f.count("*") / model_train_data_final.count()) * 100, 2).alias("percentage")
).display()

test_data_final.groupBy("target_label").agg(
    f.count("*").alias("hh_count"),
    f.round((f.count("*") / 
test_data_final.count()) * 100, 2).alias("percentage")
).display()


In [0]:
# Ordinal Encoding for Segmentations

def ordinal_transform(df):
    # Use string representations for replacement values
    seg_mapping = {"Unassigned": "0", "L": "1", "M": "2", "H": "3"}
    dig_mapping = {"New": "0", "Unassigned": "1", "L": "2", "M": "3", "H": "4"}

    seg_columns = ["price_dim_seg", "health_dim_seg", "convenience_dim_seg", "variety_seeking_dim_seg"]
    all_target_cols = seg_columns + ["dig_eng_seg_code"]

    for col in all_target_cols:
        df = df.withColumn(col, f.trim(f.col(col)))

    df = df.replace(to_replace=seg_mapping, subset=seg_columns)
    df = df.replace(to_replace=dig_mapping, subset=["dig_eng_seg_code"])
    
    for col in all_target_cols:
        df = df.withColumn(col, f.col(col).cast("integer"))

    return df

In [0]:
model_train_data_final = ordinal_transform(model_train_data_final)
test_data_final = ordinal_transform(test_data_final)

model_train_data_final.display()

In [0]:
# Create H2O instance 

h2o.init()

In [0]:
# Initialize H2O Context
hc = H2OContext.getOrCreate()

# Convert PySpark DataFrames to H20Frame

# Need to downsample - tooooo many HHs
positives = model_train_data_final.filter(f.col("target_label") == "1")
negatives = model_train_data_final.filter(f.col("target_label") == "0").sample(False, 0.20, seed=8451)
model_train_data_final_downsampled = positives.union(negatives)

h2o_train = hc.asH2OFrame(model_train_data_final_downsampled)
h2o_test = hc.asH2OFrame(test_data_final)

In [0]:
model_train_data_final.columns

In [0]:
# Convert enum to int for continuous variables (or dimensionality curse will kill h2o cluster)
numeric_cols = ['total_net_spend',
 'total_trips',
 'halloween_2024_kpf_net_spend',
 'halloween_2024_kpf_trips',
 'holiday_2024_kpf_net_spend',
 'holiday_2024_kpf_trips',
 'spr_eas_mothers_2025_kpf_net_spend',
 'spr_eas_mothers_2025_kpf_trips',
 'fathers_summer_grad_2025_kpf_net_spend',
 'fathers_summer_grad_2025_kpf_trips',
 'vtines_2025_kpf_net_spend',
 'vtines_2025_kpf_trips',
 'halloween_2024_kpf_spend_pct',
 'holiday_2024_kpf_spend_pct',
 'spr_eas_mothers_2025_kpf_spend_pct',
 'fathers_summer_grad_2025_kpf_spend_pct',
 'vtines_2025_kpf_spend_pct',
 'total_fuel_points',
 'halloween_2024_kpf_fuel_points',
 'holiday_2024_kpf_fuel_points',
 'spr_eas_mothers_2025_kpf_fuel_points',
 'fathers_summer_grad_2025_kpf_fuel_points',
 'vtines_2025_kpf_fuel_points',
 'halloween_2024_kpf_fuel_points_pct',
 'holiday_2024_kpf_fuel_points_pct',
 'spr_eas_mothers_2025_kpf_fuel_points_pct',
 'fathers_summer_grad_2025_kpf_fuel_points_pct',
 'vtines_2025_kpf_fuel_points_pct',
 'total_greeting_card_spend',
 'avg_greeting_card_spend_per_trip',
 'avg_days_between_greeting_card_buys',
 'greeting_card_trips_count']

for col in numeric_cols:
    h2o_train[col] = h2o_train[col].asnumeric()
    h2o_test[col] = h2o_test[col].asnumeric()

In [0]:
# Conversion Check
print(h2o_train.types)
h2o_train.describe()

# small subset for example ...
suspect_cols = ["total_net_spend", "total_trips", "total_fuel_points", 
                "holiday_2024_kpf_net_spend", "holiday_2024_kpf_trips"]
for col in suspect_cols:
    print(col, h2o_train[col].nlevels())

In [0]:
# Run AutoML excluding Target Label and EHHN
target_col = "target_label" 
id_col = "ehhn"

h2o_train[target_col] = h2o_train[target_col].asfactor()
feature_cols = [col for col in h2o_train.columns if col not in [target_col, id_col]]

# Reduced max_models and cross-validation folds to prevent OOM
auto_ml = H2OAutoML(
    max_models=10, 
    nfolds=0, 
    seed=8451, 
    exclude_algos=['StackedEnsemble', 'DeepLearning'])


auto_ml.train(
    x=feature_cols, 
    y=target_col, 
    training_frame=h2o_train
)

h2o.cluster_status()

In [0]:
#h2o.cluster_status()

In [0]:
top_model = auto_ml.get_best_model()

In [0]:
# display metrics for top model
top_model.model_performance(h2o_test)

# AUC 

Top performing model across multiple runs is DRF (distributed random forest), tldr: ensembling method where hundreds of random forests generated via bootstrapping ran in parallel, and average of probabilities taken for classification.
- Solid AUC: Model is decently good at differentiating between classes. In addition, the model is ~ 80% likely to assign a higher acquisition probability score to a true holiday gift buyer than to a non-gift buyer.
- Small RMSE and MSE which is good, but also a bit misleading due to class imbalance. The model is very good at identifying which HHs will not become 3P Gift HHs, but there are also a lot more "0" label HHs than "1" so this low MSE is lowkey almost artificial in a way ..
- I think its okay here to ignore the confusion matrix and recall metrics, due to the high clance imbalance. Since I used sample_mart = False, even with downsampling, the number of HHs that did not become gift HHs during Nov-Dec far outnumber those who did (7.8 Million to 78K). But AUC tells us that the model for the most part scores a "1" HH higher than a "0" HH, which I think is what we care ab.
- Looking at which features impact the model predictions, as well as comparing control sales of the HHs predicted by the model VS. true acqusition cell will probably give us a better idea of how it does.

In [0]:
# View leaderboard
leaderboard = auto_ml.leaderboard
leaderboard.head()

# rmse, mse: avg of sqd differences between truth and prediction .. but i think this model is very good at
# identifying "0" label HHs (non-gift), which makes the mse number low 

# Gains_lift: how many times better than random selection that group performs. Gains table already generated from model_performance()
#gains_lift = top_model.gains_lift(h2o_test)
#print(gains_lift)

In [0]:
# SHAP for feature importance + direction
#contributions = top_model.predict_contributions(h2o_test)
#contributions_df = contributions.as_data_frame()

h2o_test_sample = h2o_test.split_frame(ratios=[0.30], seed=8451)[0]
top_model.shap_summary_plot(h2o_test_sample)

SHAP feature summary plot key takeaways:
- Seems like all features that I used here that are pictured have some impact / drives model predictions some way, which is a good sign.
- Fathers / Summer (the season right before Halloween if we exclude fall) seems to also be a big factor, as well as 2024 halloween. This tells me the features are working as intended.
- Digital engagement is interesting - since it was not a big factor at all for Holiday, but seems to be for Halloween.
- The majority of seasonal fuel point and visits related features seem to drive positive (new gift HH) predictions. Red points indicate that the higher this feature value is, it points towards positive predictions. TLDR: more fuel point redemptions / net spend / higher spend pct during key seasons means higher chance of a HH being a Holiday gift buyer.
- Lower total trips -> higher likelihood of a HH not being a gift buyer is the most impactful singular feature .. which makes sense?
- Order of importance of features might shift slightly as I downsample less, but trends should stay relatively the same.

In [0]:
# Prediction on h2o_test, not h2o_train 
preds = top_model.predict(h2o_test)

# Get conversion Rate 
# Get Number and Which HHs that we predicted that acquisition did not

# Get HHs and their likelihood scores + classification
scored = h2o_test["ehhn"].cbind(preds)
scored.columns = ["ehhn", "predict", "p0", "p1"]
scored.head(20)

In [0]:
scored_spark = hc.asSparkFrame(scored)
scored_spark.coalesce(1).write.mode("overwrite").option("header", "true").csv('abfss://data-shuttle@sa8451denblrdatamoverdev.dfs.core.windows.net/d199104/lookalike/acquisition_lookalike__halloween_2025_predicted_HHS_result')

In [0]:
# Scored Test Set HHs (7M Test, 31M Control)
pred_gift_hhs = spark.read.csv('abfss://data-shuttle@sa8451denblrdatamoverdev.dfs.core.windows.net/d199104/lookalike/acquisition_lookalike__halloween_2025_predicted_HHS_result', header=True, inferSchema=True)
hh_count = pred_gift_hhs.select("ehhn").distinct().count()
print(f"Count of HHs: {hh_count}")

pred_gift_hhs_sorted = pred_gift_hhs.orderBy(f.col("p1").desc())
#pred_gift_hhs_filtered = pred_gift_hhs_sorted.filter((f.col("p1") < 0.5) & (f.col("predict") == "1"))
display(pred_gift_hhs_sorted)

In [0]:
p1_hhs = pred_gift_hhs_sorted.filter(f.col("predict") == "1")
hh_count = p1_hhs.select("ehhn").distinct().count()
print(f"Count of HHs: {hh_count}")

In [0]:
# Verifying Top Performer Product Group to use for pre_post_period_parquet
kpf_mhtv_mmci_data = spark.read.csv('abfss://data-shuttle@sa8451denblrdatamoverdev.dfs.core.windows.net/d199104/kpf_dashboard/closed_loop_summary_tab_INTERMEDIATE.csv', header=True, inferSchema=True)
kpf_mhtv_mmci_data.filter(f.col("campaign_id").isin("155120", "150472", "150959")).display()

In [0]:
# Post Period Sales per Test HH via Pre_post_metrics parquet
pre_post = spark.read.parquet(
  f'abfss://measure@sa8451camprd.dfs.core.windows.net/intermediate/attribution/pre_post_period_metrics/campaign_type=*/campaign_id=150959').filter(
  f.col("hhgroup") == "TEST")
pre_post.display()
pre_post.count()

1. Test to see if there is a significant difference between HHs that the model thinks are good VS. HHs that the model thinks are not good WITHIN the acquisition cell.
- Stat test on Acquisition Cell, but Cell 1: HHs that the model predicted were "good", and Cell 2: HHs that the model predicted were "not good". This way, both samples come from the same (sufficiently large sample size) population, and there is no overlap of HHs between samples.
- We would also be comparing a fully "treated" group AKA acquisition cell with a mixed group (some treated, some not treated, such as HHs the model think are good that are not in an acquisition cell), which is invalid.
- If we remove "good" model HHs from the acquisition cell (we originally planned on doing good model HHs vs. acquisition cell HHs excluding good model HHs), we might artificially be deflating post period sales from a sample, which introduces bias.

I DONT think we should we should filter out Hhs with 0 post period spend when running stat tests. i feel like we would lose too much info (especially because a lot of hhs in the pre post parq have 0/null/na post period spend) Since there are a lot of 0s:
- First, compare conversion rate of the two samples. is there a sig. diff here? 
- Out of those who converted (post period sales > 0), is their post period sales statistically significant?

NOTE to self: Previously thought about comparing post period sales of: "Good" HHs that the model predicts VS. Acquisition Cell HHs excluding Good HHs. 
- Both samples wont come from the same population and are not matched groups, so i dont think a stat test will work here. "Good HHs" includes any hh in the test set that the model thinks will be a gift buyer, which is a different population from the acquisition cell. Different populations means we cant split the pop into two samples ..  so idt a stat test will rly tell us anything.
- Removing overlaping HHs from acq. cell before testing artificially deflates this sample so prob not valid

In [0]:
pred_gift_hhs_fixed = pred_gift_hhs.withColumn("ehhn", f.trim(f.col("ehhn").cast("string")))
pre_post_fixed = pre_post.withColumn("hshd_code", f.trim(f.col("hshd_code").cast("string")))

In [0]:
# All HHs that we targeted via acquisition cell
targeted_pre_post_hhs = pre_post_fixed.filter(
    (f.col("targeting_sse").isin("New Acquisition Cell V1", "New Acquisition Cell V2", "New Acquisition Cell V3")) | 
    (f.col("targeting_push").isin("New Acquisition Cell V1", "New Acquisition Cell V2", "New Acquisition Cell V3"))
)

print(f"Targeted HHs count: {targeted_pre_post_hhs.count():,}")

# Join acquisition cell targeted HHs with HH model scores (from the test dataset). 
# Obviously we will lose lots of HHs from this join because the test dataset we use to validate the model 
# is only ~20% of the entire training dataset that we used to tune the h2o model (but easily big enough sample size)
targeted_pre_post_hhs_scored = pred_gift_hhs_fixed.join(
    targeted_pre_post_hhs,
    pred_gift_hhs_fixed["ehhn"] == targeted_pre_post_hhs["hshd_code"],
    how="inner"
)

print(f"Scored Targeted HHs count: {targeted_pre_post_hhs_scored.count():,}")

targeted_pre_post_hhs_scored_good = targeted_pre_post_hhs_scored.filter(f.col("predict") == "1")
targeted_pre_post_hhs_scored_bad = targeted_pre_post_hhs_scored.filter(f.col("predict") == "0")


In [0]:
# Total "good" HHs, and conversion of those HHs into post period sales (meaning they have > 0 post period sales)
good_hh_sales = targeted_pre_post_hhs_scored_good.select(
    f.count("*").alias("total"),
    f.count(f.when(f.col("ol_and_3p_commodity_sept_2025_sum_sales_post") > 0, 1)).alias("converted")
).collect()[0]

good_hhs_total = good_hh_sales["total"]
good_hhs_converted = good_hh_sales["converted"]
good_hhs_non_converted = good_hhs_total - good_hhs_converted

# Total "bad" HHs, and conversion of those HHs into post period sales
bad_hh_sales = targeted_pre_post_hhs_scored_bad.select(
    f.count("*").alias("total"),
    f.count(f.when(f.col("ol_and_3p_commodity_sept_2025_sum_sales_post") > 0, 1)).alias("converted")
).collect()[0]

bad_hhs_total = bad_hh_sales["total"]
bad_hhs_converted = bad_hh_sales["converted"]
bad_hhs_non_converted = bad_hhs_total - bad_hhs_converted

# Results:
# Converted means acquisition HHs that had non-zero post period sales
print(f"Good Cell: Converted = {good_hhs_converted:,} | Non-converted = {good_hhs_non_converted:,} (Total: {good_hhs_total:,})")
print(f"Bad Cell:  Converted = {bad_hhs_converted:,} | Non-converted = {bad_hhs_non_converted:,} (Total: {bad_hhs_total:,})")

In [0]:
# run X2 test of independence to see if model scores and conversion rates are independent of each other
contingency_table = [[good_hhs_converted, good_hhs_non_converted],
               [bad_hhs_converted, bad_hhs_non_converted]]

chi2, p_value, dof, expected = stats.chi2_contingency(contingency_table)
# Expected Converted and Non Converted if independent
print("Expected counts:", expected)

# Actual
print(f"Good Cell: Converted = {good_hhs_converted:,} | Non-converted = {good_hhs_non_converted:,} (Total: {good_hhs_total:,})")
print(f"Bad Cell:  Converted = {bad_hhs_converted:,} | Non-converted = {bad_hhs_non_converted:,} (Total: {bad_hhs_total:,})")

# We see dependency
print(f"Chi-square p-value: {p_value:.6f}")

# output
#Expected counts: [[   42.19490958  6182.80509042]
# [  210.80509042 30889.19490958]]
#Good Cell: Converted = 70 | Non-converted = 6,155 (Total: 6,225)
#Bad Cell:  Converted = 183 | Non-converted = 30,917 (Total: 31,100)
#Chi-square p-value: 0.000004

In [0]:
# From those that have converted .. see if means of post period sales are different between good and bad cells
good_buyers = targeted_pre_post_hhs_scored_good.filter(
    f.col("ol_and_3p_commodity_sept_2025_sum_sales_post") > 0
).select("ol_and_3p_commodity_sept_2025_sum_sales_post").toPandas()["ol_and_3p_commodity_sept_2025_sum_sales_post"]

bad_buyers = targeted_pre_post_hhs_scored_bad.filter(
    f.col("ol_and_3p_commodity_sept_2025_sum_sales_post") > 0
).select("ol_and_3p_commodity_sept_2025_sum_sales_post").toPandas()["ol_and_3p_commodity_sept_2025_sum_sales_post"]

# t-test (Assumes uneven sample sizes + variance), tests for stat. sig difference in means of 2 groups
t_stat, p_val_welch = stats.ttest_ind(good_buyers, bad_buyers, equal_var=False)
print(f"Welch's t-test p-value: {p_val_welch:.6f}")

print(f"Good buyers - mean: ${good_buyers.mean():.2f}, median: ${good_buyers.median():.2f}, n={len(good_buyers)}")
print(f"Bad buyers  - mean: ${bad_buyers.mean():.2f}, median: ${bad_buyers.median():.2f}, n={len(bad_buyers)}")

# in theory this still means higher conversion + same ish post period spend -> uplift
# also makes sense cuz model is identifying who is likely to convert to a gift buyer, not who looks like an alr. established buyer

In [0]:
# everyone NOT in the acquisition cell
non_targeted = pre_post.join(
    targeted_pre_post_hhs.select("hshd_code"),
    on="hshd_code",
    how="left_anti"
)

# join to model scores
non_targeted_scored = pred_gift_hhs.join(
    non_targeted,
    pred_gift_hhs["ehhn"] == non_targeted["hshd_code"],
    how="inner"
)

group_B_good = non_targeted_scored.filter(f.col("predict") == "1")
group_D_bad = non_targeted_scored.filter(f.col("predict") == "0")

print(f"Group B (model-good, non-targeted): {group_B_good.count():,}")
print(f"Group D (model-bad, non-targeted): {group_D_bad.count():,}")

In [0]:
# Conversion rates of
good_hh_sales_nontarget = group_B_good.select(
    f.count("*").alias("total"),
    f.count(f.when(f.col("ol_and_3p_commodity_sept_2025_sum_sales_post") > 0, 1)).alias("converted")
).collect()[0]

good_hhs_total_nontarget = good_hh_sales_nontarget["total"]
good_hhs_converted_nontarget = good_hh_sales_nontarget["converted"]
good_hhs_non_converted_nontarget = good_hhs_total_nontarget - good_hhs_converted_nontarget

bad_hh_sales_nontarget = group_D_bad.select(
    f.count("*").alias("total"),
    f.count(f.when(f.col("ol_and_3p_commodity_sept_2025_sum_sales_post") > 0, 1)).alias("converted")
).collect()[0]

bad_hhs_total_nontarget = bad_hh_sales_nontarget["total"]
bad_hhs_converted_nontarget = bad_hh_sales_nontarget["converted"]
bad_hhs_non_converted_nontarget = bad_hhs_total_nontarget - bad_hhs_converted_nontarget

print(f"Good Cell (Non-Target): Converted = {good_hhs_converted_nontarget:,} | Non-converted = {good_hhs_non_converted_nontarget:,} (Total: {good_hhs_total_nontarget:,})")
print(f"Bad Cell (Non-Target):  Converted = {bad_hhs_converted_nontarget:,} | Non-converted = {bad_hhs_non_converted_nontarget:,} (Total: {bad_hhs_total_nontarget:,})")

In [0]:
# run X2 test of independence to see if model scores and conversion rates are independent
contingency_nontarget = [
    [good_hhs_converted_nontarget, good_hhs_non_converted_nontarget],
    [bad_hhs_converted_nontarget, bad_hhs_non_converted_nontarget]
]

chi2, p_val_nontarget, dof, expected = stats.chi2_contingency(contingency_nontarget)
# Expected Converted and Non Converted if model scores and conversion rates were independent
print("Expected counts:", expected)

# Actual
print(f"Good Cell (Non-Target): Converted = {good_hhs_converted_nontarget:,} | Non-converted = {good_hhs_non_converted_nontarget:,} (Total: {good_hhs_total_nontarget:,})")
print(f"Bad Cell (Non-Target):  Converted = {bad_hhs_converted_nontarget:,} | Non-converted = {bad_hhs_non_converted_nontarget:,} (Total: {bad_hhs_total_nontarget:,})")

# We see dependency (again)
print(f"Chi-square p-value: {p_val_nontarget:.6f}")

In [0]:
# From those that have converted .. see if means of post period sales are different between good and bad cells
good_buyers_nontarget = group_B_good.filter(
    f.col("ol_and_3p_commodity_sept_2025_sum_sales_post") > 0
).select("ol_and_3p_commodity_sept_2025_sum_sales_post").toPandas()["ol_and_3p_commodity_sept_2025_sum_sales_post"]

bad_buyers_nontarget = group_D_bad.filter(
    f.col("ol_and_3p_commodity_sept_2025_sum_sales_post") > 0
).select("ol_and_3p_commodity_sept_2025_sum_sales_post").toPandas()["ol_and_3p_commodity_sept_2025_sum_sales_post"]

# t-test (Assumes uneven sample sizes + variance), tests for stat. sig difference in means of 2 groups
t_stat, p_val_welch_nontarget = stats.ttest_ind(good_buyers_nontarget, bad_buyers_nontarget, equal_var=False)
print(f"Welch's t-test p-value: {p_val_welch_nontarget:.6f}")

print(f"Good buyers (non-target) - mean: ${good_buyers_nontarget.mean():.2f}, median: ${good_buyers_nontarget.median():.2f}, n={len(good_buyers_nontarget)}")
print(f"Bad buyers (non-target)  - mean: ${bad_buyers_nontarget.mean():.2f}, median: ${bad_buyers_nontarget.median():.2f}, n={len(bad_buyers_nontarget)}")